# Closed-Loop Gain Recovery Test

Synthetic closed-loop demonstration of the `newnucal` calibration pipeline:

1. Build a HERA-like array and a random chromatic sky model
2. Simulate visibilities with `ForwardModel` at several times (Earth rotation) and frequencies
3. Apply known per-frequency gain degeneracies (amplitude, phase, phase gradient)
4. Recover gains with the sky held fixed — checking we get back what we put in
5. Show what happens when the sky is also perturbed and we optimize jointly

The key physics being tested: the DPSS spectral constraints on the sky/beam model, combined with Earth rotation that decorrelates sky pixels from beam pixels across time, provide enough information to disentangle per-frequency gain corrections from the sky signal.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from astropy.time import Time
from astropy.coordinates import EarthLocation
import healpy

jax.config.update("jax_enable_x64", False)  # float32 throughout

from newnucal import HERAArray, BeamModel, ForwardModel, Calibrator, apply_gains, init_gain_params
from newnucal.dpss import dpss_matrix
from newnucal.simulate import compute_rotation_matrices

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110})

## 1. Array, frequency, and time setup

In [ ]:

# --- Array ---
array = HERAArray.from_hex(hexnum=4, sep=14.6)
print(f"Antennas: {array.nants},  Baselines: {array.nbls}")

# --- Frequencies ---
nfreq = 64
freqs = np.linspace(50e6, 225e6, nfreq)  # Hz

# --- Times ---
from astropy.time import Time
import astropy.units as u

hera_loc = EarthLocation(lat=-30.7215 * u.deg, lon=21.4283 * u.deg, height=1073.0 * u.m)
t0 = Time("2023-03-21T04:00:00", scale="utc")
ntime = 8
dt = 60 / ntime * u.min
times = t0 + np.arange(ntime) * dt
print(f"Times: {ntime},  span: {(times[-1] - times[0]).to(u.min):.1f}")

# --- Rotation matrices ---
from newnucal.simulate import compute_rotation_matrices
rot_matrices = compute_rotation_matrices(times, hera_loc)
print(f"rot_matrices shape: {rot_matrices.shape}")


## 2. Beam and sky model

In [ ]:
sky_nside = 32
beam_nside = 16

# Beam: Airy disk (HERA dish diameter 14.6 m), DPSS eta_max = 20 ns
beam_model = BeamModel(nside=beam_nside, freqs=freqs, eta_max=20e-9)
print(f"Beam DPSS modes: {beam_model.A_beam.shape[1]}")

# Sky: random power-law HEALPix map
npix_sky = healpy.nside2npix(sky_nside)
rng = np.random.default_rng(42)

# Build a smooth chromatic sky: per-pixel spectral index drawn from N(-0.7, 0.1)
ref_freq = 150e6
spectral_indices = rng.normal(-0.7, 0.1, npix_sky).astype(np.float32)
ref_flux = rng.exponential(scale=1.0, size=npix_sky).astype(np.float32)
# flux_true[npix, nfreq]
flux_true = ref_flux[:, None] * (freqs[None, :] / ref_freq) ** spectral_indices[:, None]

# Project onto sky DPSS basis
sky_eta_max = 40e-9  # ns — wider than beam because sky has steeper spectral structure
from newnucal.dpss import dpss_matrix, dpss_project
A_sky = dpss_matrix(freqs, sky_eta_max)
print(f"Sky DPSS modes: {A_sky.shape[1]}")

sky_coeffs_true = jnp.array(dpss_project(flux_true, A_sky), dtype=jnp.float32)
print(f"sky_coeffs_true shape: {sky_coeffs_true.shape}")


## 3. Simulate true visibilities and apply known gain perturbations

In [ ]:
fwd = ForwardModel(array, sky_nside, beam_model, freqs, eps=1e-5)
fwd.set_sky_dpss(A_sky)

print("Simulating true visibilities (this compiles the JIT on first run)...")
vis_true = fwd.simulate(sky_coeffs_true, jnp.array(rot_matrices))
print(f"vis_true shape: {vis_true.shape},  dtype: {vis_true.dtype}")
print(f"Mean |vis|: {float(jnp.abs(vis_true).mean()):.4f}")

In [ ]:
# --- True gain perturbations ---
# log_amp: smooth ~5% amplitude variation across band
true_log_amp = 0.05 * np.cos(2 * np.pi * np.arange(nfreq) / nfreq).astype(np.float32)

# phase: smooth ~0.15 rad variation across band
true_phase = 0.15 * np.sin(2 * np.pi * np.arange(nfreq) / nfreq).astype(np.float32)

# phi: small phase gradients, ~1e-4 rad/m, smooth across band
true_phi = np.zeros((2, nfreq), dtype=np.float32)
true_phi[0] = 1e-4 * np.cos(2 * np.pi * np.arange(nfreq) / nfreq)  # East
true_phi[1] = 5e-5 * np.sin(2 * np.pi * np.arange(nfreq) / nfreq)  # North

true_log_amp_j = jnp.array(true_log_amp)
true_phase_j   = jnp.array(true_phase)
true_phi_j     = jnp.array(true_phi)

vis_data = apply_gains(vis_true, true_log_amp_j, true_phase_j, true_phi_j,
                       jnp.array(array.bls, dtype=jnp.float32))

print(f"vis_data shape: {vis_data.shape}")
print(f"RMS gain amplitude perturbation: {float(jnp.exp(jnp.array(true_log_amp_j)).std()):.4f}")

## 4. Stage 1: Recover gains with sky held fixed (true sky)

In [ ]:

cal = Calibrator(
    array=array,
    beam_model=beam_model,
    sky_nside=sky_nside,
    sky_eta_max=sky_eta_max,
    freqs=freqs,
    rot_matrices=rot_matrices,
    data=vis_data,
    eps=1e-5,
)

gain_params0 = init_gain_params(nfreq)

print("Fitting gains only (sky = truth, held fixed)...")
gain_params_fit, loss_final = cal.fit_gains_only(
    sky_coeffs_true, gain_params0, maxiter=200, tol=1e-10
)
print(f"Final loss: {loss_final:.4e}")


## 5. Diagnostic plots: recovered vs true gains

In [ ]:
freq_mhz = freqs / 1e6

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("Stage 1: Gain recovery (sky fixed at truth)", fontsize=13)

# log_amp
ax = axes[0, 0]
ax.plot(freq_mhz, true_log_amp, "k-", lw=2, label="True")
ax.plot(freq_mhz, np.array(gain_params_fit["log_amp"]), "r--", lw=1.5, label="Recovered")
ax.set_ylabel("log_amp")
ax.set_xlabel("Frequency (MHz)")
ax.legend()
ax.set_title("Amplitude (log)")

# phase
ax = axes[0, 1]
ax.plot(freq_mhz, true_phase, "k-", lw=2, label="True")
ax.plot(freq_mhz, np.array(gain_params_fit["phase"]), "r--", lw=1.5, label="Recovered")
ax.set_ylabel("phase (rad)")
ax.set_xlabel("Frequency (MHz)")
ax.legend()
ax.set_title("Phase")

# phi_x (East gradient)
ax = axes[1, 0]
ax.plot(freq_mhz, true_phi[0], "k-", lw=2, label="True")
ax.plot(freq_mhz, np.array(gain_params_fit["phi"][0]), "r--", lw=1.5, label="Recovered")
ax.set_ylabel("phi_E (rad/m)")
ax.set_xlabel("Frequency (MHz)")
ax.legend()
ax.set_title("Phase gradient (East)")

# phi_y (North gradient)
ax = axes[1, 1]
ax.plot(freq_mhz, true_phi[1], "k-", lw=2, label="True")
ax.plot(freq_mhz, np.array(gain_params_fit["phi"][1]), "r--", lw=1.5, label="Recovered")
ax.set_ylabel("phi_N (rad/m)")
ax.set_xlabel("Frequency (MHz)")
ax.legend()
ax.set_title("Phase gradient (North)")

plt.tight_layout()
plt.show()

# Print residuals
print("Recovery residuals (RMS):")
print(f"  log_amp: {float(jnp.sqrt(jnp.mean((gain_params_fit['log_amp'] - true_log_amp_j)**2))):.2e}")
print(f"  phase:   {float(jnp.sqrt(jnp.mean((gain_params_fit['phase']   - true_phase_j)**2))):.2e}")
print(f"  phi_E:   {float(jnp.sqrt(jnp.mean((gain_params_fit['phi'][0]  - true_phi_j[0])**2))):.2e}")
print(f"  phi_N:   {float(jnp.sqrt(jnp.mean((gain_params_fit['phi'][1]  - true_phi_j[1])**2))):.2e}")

## 6. Stage 2: Joint sky + gain recovery (perturbed sky start)

Start the sky from a slightly perturbed version of the truth and jointly optimize sky coefficients and gains. This tests whether the optimization can disentangle the two.

In [ ]:

# Perturb sky by 10% Gaussian noise on coefficients
rng2 = np.random.default_rng(99)
sky_coeffs_perturbed = sky_coeffs_true + 0.10 * jnp.array(
    rng2.standard_normal(sky_coeffs_true.shape).astype(np.float32)
) * float(jnp.abs(sky_coeffs_true).mean())

params_joint0 = {
    "sky_coeffs": sky_coeffs_perturbed,
    **init_gain_params(nfreq),
}

print("Joint sky+gain fit (L-BFGS)...")
params_joint, loss_lbfgs = cal.fit_lbfgs(params_joint0, maxiter=60, tol=1e-7)
print(f"  L-BFGS loss: {loss_lbfgs:.4e}")

print("Joint sky+gain fine-tune (Adam)...")
params_joint, loss_adam = cal.fit_optax(params_joint, maxiter=120, lr=1e-2, verbose=False)
print(f"  Adam loss:   {loss_adam:.4e}")


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
fig.suptitle("Stage 2: Joint gain recovery (10% sky perturbation)", fontsize=13)

for (ax, key, true_val, label) in [
    (axes[0, 0], "log_amp", true_log_amp, "log_amp"),
    (axes[0, 1], "phase",   true_phase,   "phase (rad)"),
]:
    ax.plot(freq_mhz, true_val, "k-", lw=2, label="True")
    ax.plot(freq_mhz, np.array(params_joint[key]), "r--", lw=1.5, label="Recovered (joint)")
    ax.set_ylabel(label)
    ax.set_xlabel("Frequency (MHz)")
    ax.legend(fontsize=8)

axes[1, 0].plot(freq_mhz, true_phi[0], "k-", lw=2, label="True")
axes[1, 0].plot(freq_mhz, np.array(params_joint["phi"][0]), "r--", lw=1.5, label="Recovered")
axes[1, 0].set_ylabel("phi_E (rad/m)")
axes[1, 0].set_xlabel("Frequency (MHz)")
axes[1, 0].legend(fontsize=8)
axes[1, 0].set_title("Phase gradient (East)")

axes[1, 1].plot(freq_mhz, true_phi[1], "k-", lw=2, label="True")
axes[1, 1].plot(freq_mhz, np.array(params_joint["phi"][1]), "r--", lw=1.5, label="Recovered")
axes[1, 1].set_ylabel("phi_N (rad/m)")
axes[1, 1].set_xlabel("Frequency (MHz)")
axes[1, 1].legend(fontsize=8)
axes[1, 1].set_title("Phase gradient (North)")

plt.tight_layout()
plt.show()

print("Joint recovery residuals (RMS):")
print(f"  log_amp: {float(jnp.sqrt(jnp.mean((params_joint['log_amp'] - true_log_amp_j)**2))):.2e}")
print(f"  phase:   {float(jnp.sqrt(jnp.mean((params_joint['phase']   - true_phase_j)**2))):.2e}")
print(f"  phi_E:   {float(jnp.sqrt(jnp.mean((params_joint['phi'][0]  - true_phi_j[0])**2))):.2e}")
print(f"  phi_N:   {float(jnp.sqrt(jnp.mean((params_joint['phi'][1]  - true_phi_j[1])**2))):.2e}")

# Also compare sky recovery
sky_rec = params_joint["sky_coeffs"] @ jnp.array(A_sky).T  # (npix, nfreq)
sky_rms_err = float(jnp.sqrt(jnp.mean((sky_rec - jnp.array(flux_true))**2)))
sky_rms_true = float(jnp.sqrt(jnp.mean(jnp.array(flux_true)**2)))
print(f"\n  sky flux RMS error / RMS true: {sky_rms_err / sky_rms_true:.3f}")
